In [ ]:
import os
import sys

import tensorflow as tf
import yaml

sys.path.append(os.path.abspath("../"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import glob

import matplotlib.pyplot as plt
import numpy as np
import sklearn

from train.models import FullModel
from util import dataset as u_dataset
from util import dataset_io as u_dataset_io
from util import metrics as u_metrics

dataset_utils = u_dataset.DatasetUtils(u_dataset.DatasetConfig())

In [ ]:
model_name = "20260203-163416.keras"
path_to_models = "../../models/finished/"

model_timestamp = model_name.split(".")[0]

In [ ]:
# Get encoder config
def load_config(config_path):
    with open(config_path) as f:
        config = yaml.safe_load(f)
    return config


config_dir = f"../../logs/fit/{model_timestamp}/"
config = load_config(config_dir + "config.yaml")

encoder_architecture = config["model"]["encoder"]["architecture"]
classifier_architecture = config["model"]["classifier"]["architecture"]
categories_config = config["categories"]

#### Load Test Data


In [ ]:
path_to_test_data = glob.glob("../../data/tfrecords/test_ds_v3*.tfrecords")
test_ds = u_dataset_io.get_dataset(path_to_test_data)

batch_size = len(list(test_ds.as_numpy_iterator()))  # only one batch
test_ds = test_ds.batch(batch_size, drop_remainder=False)

#### Load Model


In [ ]:
model = FullModel.load(
    encoder_architecture,
    classifier_architecture,
    input_dims=config["model"]["encoder"]["input_dims"],
    filepath=path_to_models,
    filename=model_name,
    n_context=config["model"]["encoder"]["n_context"],
    only_train_encoder=config["model"]["encoder"]["only_train_encoder"],
    classifier_offsets=config["model"]["classifier"]["with_offsets"],
    encoder_only=False,
    verbose=True,
    n_meta=config["model"]["classifier"]["n_meta"],
    encoder_use_batch_norm=config["model"]["encoder"]["use_batch_norm"],
    classifier_use_batch_norm=config["model"]["classifier"]["use_batch_norm"],
    categories_config=config["categories"],
)
model.compile(optimizer=tf.keras.optimizers.Adam(), jit_compile=False)


#### Predict Data


In [ ]:
groundtruth_dataset = []

for x in test_ds:
    groundtruth_dataset.append(x)


# Take only image, camera, intrinsics for input
input_data = test_ds.map(lambda x: (x["image"], x["camera"], x["intrinsics"]))

# Convert tf.Data.Dataset into list
input_list = []
for x, y, z in input_data:
    input_list.append((x, y, z))

model.run_eagerly = True

predictions = [model.predict(x=batch) for batch in input_list]
metrics_list = [model.evaluate(x=batch, return_dict=True) for batch in groundtruth_dataset]

#### Evaluate Models


In [ ]:
# Calculate mean for metrics over each batch
keys = metrics_list[0].keys()
stacked_values = {key: tf.stack([metrics[key] for metrics in metrics_list]) for key in keys}
metrics_mean = {key: float(tf.reduce_mean(stacked_values[key])) for key in keys}

# Print Metrics
for key, value in metrics_mean.items():
    print(f"{key}: {value:.4f}")

### Accuracy, Precision, Recall


In [ ]:
# def reshape_tensors(object_tensors: list, reshape=True):
#     """Combine a list of tensors batches into a single tensor.

#     Args:
#         object_tensors: list of tensor batches
#         reshape: . Defaults to True.

#     Returns:
#         A single tensor.
#     """
#     stacked_tensors = {
#         key: tf.stack([object_tensor[key] for object_tensor in object_tensors])
#         for key in object_tensors[0]
#     }
#     if reshape:
#         return {
#             key: tf.reshape(tensor, (-1, *tensor.shape[2:]))
#             for key, tensor in stacked_tensors.items()
#         }
#     else:
#         return stacked_tensors

In [ ]:
def get_metrics(thresholds: dict, encoder_threshold: float = 0.1) -> dict:
    metrics = {}

    for object in u_dataset.CategoryNames:
        if object.value not in predictions[0]["results"]:
            print(f"No {object.value} in results.")
            continue

        if object.name == u_dataset.CategoryNames.INTERSECTIONS.name:
            results = [
                u_metrics.calculate_multiclass_metrics(
                    predictions[0]["results"][object.value],
                    groundtruth_dataset[0][object.value],
                    cla,
                    encoder_threshold,
                )
                for cla in thresholds[object.value]
            ]
        else:
            results = [
                u_metrics.calculate_binary_metrics(
                    predictions[0]["results"][object.value],
                    groundtruth_dataset[0][object.value],
                    encoder_threshold,
                    cla,
                    True,
                )
                for cla in thresholds[object.value]
            ]

        if len(thresholds[object.value]) == 1:
            metrics[object.value] = results[0]
        else:
            metrics[object.value] = results

    return metrics


encoder_threshold = 0.1
threshold_range = np.linspace(0, 1, 40, endpoint=True)
classifier_threshold_ranges = {
    u_dataset.CategoryNames.BALL.value: threshold_range,
    u_dataset.CategoryNames.PENALTYMARK.value: threshold_range,
    u_dataset.CategoryNames.INTERSECTIONS.value: threshold_range,
}

metrics_threshold_range = get_metrics(classifier_threshold_ranges, encoder_threshold)

#### Choose Correct Threshold


In [ ]:
classifier_thresholds = {
    u_dataset.CategoryNames.BALL.value: [0.9],
    u_dataset.CategoryNames.PENALTYMARK.value: [0.9],
    u_dataset.CategoryNames.INTERSECTIONS.value: [0.99],
}

for object in u_dataset.CategoryNames:
    if object.value not in metrics_threshold_range:
        continue

    precision_cla = [x["precision_pooled"] for x in metrics_threshold_range[object.value]]
    recall_cla = [x["recall_pooled"] for x in metrics_threshold_range[object.value]]

    plt.plot(threshold_range, precision_cla, label="Precision")
    plt.plot(threshold_range, recall_cla, label="Recall")
    plt.title(
        f"Precision and Recall Curve for {object.value}. Threshold {classifier_thresholds[object.value]}"
    )
    plt.axvline(classifier_thresholds[object.value], ls="dotted", color="r")
    plt.xlabel("Threshold")
    plt.ylabel("Precision/Recall")
    plt.legend()
    plt.show()

In [ ]:
metrics_chosen_threshold = get_metrics(classifier_thresholds, encoder_threshold)

In [ ]:
for object in u_dataset.CategoryNames:
    if object.value not in metrics_chosen_threshold:
        continue

    confusion_matrix = metrics_chosen_threshold[object.value]["confusion_matrix"]
    confusion_matrix_pooled = metrics_chosen_threshold[object.value]["confusion_matrix_pooled"]

    print(f"==== {object.value.capitalize()} ====")
    tf.print("Precision: ", metrics_chosen_threshold[object.value]["precision_pooled"])
    tf.print("Recall: ", metrics_chosen_threshold[object.value]["recall_pooled"])
    tf.print("fp rate:", metrics_chosen_threshold[object.value]["fp_rate"])
    tf.print("fn rate: ", metrics_chosen_threshold[object.value]["fn_rate"])

    if object == u_dataset.CategoryNames.INTERSECTIONS:
        sklearn.metrics.ConfusionMatrixDisplay(
            confusion_matrix, display_labels=np.array(["None", "L", "T", "X"])
        ).plot(cmap=plt.cm.viridis)
        plt.title(
            f"Confusion Matrix for {object.value.capitalize()}. Threshold {classifier_thresholds[object.value][0]}"
        )

    sklearn.metrics.ConfusionMatrixDisplay(
        confusion_matrix_pooled, display_labels=np.array(["True", "False"])
    ).plot(cmap=plt.cm.viridis)
    plt.title(
        f"Confusion Matrix for {object.value.capitalize()}. Threshold {classifier_thresholds[object.value][0]}"
    )

    plt.show()

#### Write Metrics into a log file


In [ ]:
metrics_log = {"metrics": metrics_mean, "evaluation": {}}

for object in u_dataset.CategoryNames:
    if object.value not in metrics_chosen_threshold:
        continue

    metrics_log["evaluation"][object.value] = {
        "encoder_threshold": encoder_threshold,
        "classifier_threshold": classifier_thresholds[object.value],
        "precision": float(metrics_chosen_threshold[object.value]["precision_pooled"]),
        "recall": float(metrics_chosen_threshold[object.value]["recall_pooled"]),
        "confusion_matrix_pooled": (
            metrics_chosen_threshold[object.value]["confusion_matrix_pooled"]
        ).tolist(),
        "confusion_matrix": (metrics_chosen_threshold[object.value]["confusion_matrix"]).tolist(),
        "fp_rate": float(
            confusion_matrix[0][1] / (confusion_matrix[0][1] + confusion_matrix[1][1])
        ),
        "fn_rate": float(
            confusion_matrix[1][0] / (confusion_matrix[1][0] + confusion_matrix[0][0])
        ),
        # "fp_indices": a["fp_indices"].tolist(),
        # "fn_indices": a["fn_indices"].tolist(),
    }

log_file = config_dir + "metrics.yaml"
with open(log_file, "w") as f:
    # dump config file into log
    yaml.dump(metrics_log, f, sort_keys=False, default_flow_style=False)

print(f"Configuration logged to {log_file}")

### Write predictions into file


In [ ]:
nms_iou_threshold = 0.35
save_directory = f"../../data/selected/{model_timestamp}"

for object in u_dataset.CategoryNames:
    if object.value not in predictions[0]["results"]:
        continue

    u_metrics.save_predictions(
        predictions[0]["results"][object.value],
        groundtruth_dataset[0],
        object.value,
        save_directory,
        classifier_thresholds[object.value],
        encoder_threshold,
        nms_iou_threshold,
        nms_max_output_size=config["categories"][object.value]["n_candidates"],
    )
